# 構造化データセットSFT v5.4：Strategy A（v5.2ベース + 低LRターゲット学習）

本ノートブックは、v5.4版です。  
**v5.2の学習済みモデルをベース**に、**ターゲットサンプル17件のみ**で追加SFTを行います。

### 戦略概要（Strategy A）
| 項目 | 設定値 |
|---|---|
| ベースモデル | v5.2学習済みLoRAモデル (`kmd2525/qwen3-4b-structured-output-lora-v5.2`) |
| 追加学習データ | ターゲットサンプル17件（TOML 14件、XML 3件） |
| Learning Rate | **1e-6**（非常に低い - 既存学習を破壊しない） |
| Epochs | 1 |
| LORA_R | 64（v5.2と同じ） |
| LORA_ALPHA | 128（v5.2と同じ） |

### 期待結果
- v5.2のスコア: 94.7%
- 期待スコア: 96-98%（TOML/XMLの特定エラーパターンを修正）

### ターゲットサンプルの内容
- **TOML（14件）**: inline table、配列テーブル構文、ネスト構造、YAML構文混入防止
- **XML（3件）**: 特殊文字エスケープ（`&amp;`, `&lt;`, `&gt;`, `&apos;`）

### 注意事項
- Empty Think Injection は使用しない（v5.3での失敗の原因）
- Learning Rateは非常に低く設定し、v5.2の学習を破壊しないようにする

## 1. 依存関係インストール

In [1]:
# ============================================================
# 0) 依存関係の固定（Colabの"環境ブレ"対策）
# ============================================================
!pip -q uninstall -y numpy pandas datasets trl transformers accelerate peft unsloth unsloth-zoo bitsandbytes xformers
!pip -q install "numpy==2.0.2" "pandas==2.2.2"
!pip -q install \
  "datasets==4.3.0" \
  "trl==0.24.0" \
  "transformers==4.56.2" \
  "accelerate==1.1.0" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.0"
!pip -q install "unsloth-zoo==2025.12.7" "unsloth==2025.12.7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 105.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, which is not installed.
torchtune 0.6.1 requires datasets, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.3 MB/s eta 0:00:00
ERROR: Cannot install accelerate==1.1.0 and trl==0.24.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━

In [2]:
# ============================================================
# 0.1) バージョン確認
# ============================================================
import numpy as np, pandas as pd
import datasets, trl, transformers, torch
from unsloth import FastLanguageModel

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("unsloth import OK")

/tmp/ipython-input-6212/3217508684.py:6: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
numpy 2.0.2
pandas 2.2.2
datasets 4.3.0
trl 0.24.0
transformers 4.57.3
torch 2.10.0+cu128
unsloth import OK


## 2. 設定値の定義（v5.4 Strategy A）

In [3]:
import os

# ============================================================
# v5.4 Strategy A 設定値
# ============================================================

# README 内に記載するモデルタイトル
TITLE_LINE = "qwen3-4b-structured-output-lora-v5.4"

def _getenv(name: str, default: str):
    return os.environ.get(name, default)

def _getenv_int(name: str, default: int) -> int:
    try:
        return int(os.environ.get(name, str(default)))
    except Exception:
        return default

def _getenv_float(name: str, default: float) -> float:
    try:
        return float(os.environ.get(name, str(default)))
    except Exception:
        return default

# HFリポジトリをprivateにするかどうか。(public: "0", private: "1")
HF_PRIVATE = _getenv("HF_PRIVATE", "0")

# ★ v5.4の特徴: v5.2のアダプタをマージしたモデルをベースに追加学習
# まずベースのQwenモデルを読み込み、v5.2のアダプタを適用
BASE_MODEL_ID = _getenv("SFT_BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
V52_ADAPTER_ID = _getenv("SFT_V52_ADAPTER", "kmd2525/qwen3-4b-structured-output-lora-v5.2")

# ターゲットサンプルデータセット（ローカル）
DATASET_ID = _getenv("SFT_DATASET_ID", "LOCAL:/content/v5.4_train.json")

# 学習後に保存されるLoRAアダプタの出力先（ローカル）
OUT_LORA_DIR = _getenv("SFT_OUT_LORA_DIR", "/content/lora_structeval_t_qwen3_4b_v5.4")
SEED = _getenv_int("SFT_SEED", 3407)
VAL_RATIO = _getenv_float("SFT_VAL_RATIO", 0.0)  # 17件しかないのでvalidationなし

MAX_SEQ_LEN = _getenv_int("SFT_MAX_SEQ_LEN", 1024)

# LoRA Config（v5.2と同じ設定）
LORA_R = _getenv_int("SFT_LORA_R", 64)
LORA_ALPHA = _getenv_int("SFT_LORA_ALPHA", 128)
LORA_DROPOUT = _getenv_float("SFT_LORA_DROPOUT", 0)
LORA_TARGET_MODULES = (
    _getenv("SFT_LORA_TARGET_MODULES", "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj").split(",")
)

# ★ v5.4の特徴: 非常に低いLearning Rate
NUM_TRAIN_EPOCHS = _getenv_int("SFT_EPOCHS", 1)
PER_DEVICE_TRAIN_BATCH_SIZE = _getenv_int("SFT_PER_DEVICE_TRAIN_BS", 2)
PER_DEVICE_EVAL_BATCH_SIZE = _getenv_int("SFT_PER_DEVICE_EVAL_BS", 2)
GRAD_ACCUM = _getenv_int("SFT_GRAD_ACCUM", 1)  # 17件なのでgradient accumulationは不要
LR = _getenv_float("SFT_LR", 1e-6)  # ★ 非常に低いLR
WARMUP_RATIO = _getenv_float("SFT_WARMUP_RATIO", 0.1)

# Debug / quick check
MAX_STEPS = _getenv_int("SFT_MAX_STEPS", -1)
LOGGING_STEPS = _getenv_int("SFT_LOGGING_STEPS", 1)
EVAL_STEPS = _getenv_int("SFT_EVAL_STEPS", 50)
SAVE_STEPS = _getenv_int("SFT_SAVE_STEPS", 50)
SAVE_TOTAL_LIMIT = _getenv_int("SFT_SAVE_TOTAL_LIMIT", 2)
WEIGHT_DECAY = _getenv_float("SFT_WEIGHT_DECAY", 0.05)

# CoT masking disabled（Empty Think Injection不使用）
MASK_COT = False

print("=" * 60)
print("v5.4 Strategy A 設定")
print("=" * 60)
print(f"ベースモデル: {BASE_MODEL_ID}")
print(f"v5.2アダプタ: {V52_ADAPTER_ID}")
print(f"追加学習データ: ターゲットサンプル17件")
print(f"Learning Rate: {LR} (非常に低い)")
print(f"Epochs: {NUM_TRAIN_EPOCHS}")
print(f"LORA_R: {LORA_R}, LORA_ALPHA: {LORA_ALPHA}")
print("=" * 60)

v5.4 Strategy A 設定
ベースモデル: Qwen/Qwen3-4B-Instruct-2507
v5.2アダプタ: kmd2525/qwen3-4b-structured-output-lora-v5.2
追加学習データ: ターゲットサンプル17件
Learning Rate: 1e-06 (非常に低い)
Epochs: 1
LORA_R: 64, LORA_ALPHA: 128


## 3. HuggingFace ログイン

In [4]:
from google.colab import userdata
from huggingface_hub import login, HfApi

print("Attempting login...")
MY_TOKEN = userdata.get("HF_TOKEN")
try:
    if MY_TOKEN:
        clean_token = MY_TOKEN.strip()
        login(token=clean_token)
        api = HfApi()
        print("✅ Login successful using Colab Secret (HF_TOKEN).")
    else:
        raise ValueError("HF_TOKEN is empty in Secrets.")
except Exception as e:
    print(f"⚠️ Auto-login failed: {e}")
    print("Switching to interactive login. Please enter your token below:")
    login()
    api = HfApi()

Attempting login...
✅ Login successful using Colab Secret (HF_TOKEN).


## 4. WandB ログイン（オプション）

In [5]:
import wandb

try:
    WANDB_KEY = userdata.get("WANDB_API_KEY")
    if WANDB_KEY:
        wandb.login(key=WANDB_KEY.strip())
        print("✅ WandB login successful using Colab Secret (WANDB_API_KEY).")
    else:
        raise ValueError("WANDB_API_KEY is empty in Secrets.")
except Exception as e:
    print(f"⚠️ WandB auto-login failed: {e}")
    print("Switching to interactive login...")
    wandb.login()

# プロジェクト初期化
wandb.init(
    project="structeval-sft",
    name="v5.4_strategy_a_targeted",
    config={
        "version": "v5.4",
        "strategy": "A (v5.2 base + low LR targeted)",
        "dataset": "Targeted samples (17 samples: TOML 14, XML 3)",
        "base_model": BASE_MODEL_ID,
        "v52_adapter": V52_ADAPTER_ID,
        "max_seq_len": MAX_SEQ_LEN,
        "epochs": NUM_TRAIN_EPOCHS,
        "lr": LR,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
    }
)
print(f"✅ WandB run initialized: {wandb.run.name}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kmd2525 (ken05-matuo-llm-88_llm_2025_suzuki) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ WandB login successful using Colab Secret (WANDB_API_KEY).


wandb: WARNING Failed to wrap stdout. Console logs will not be captured.
wandb: WARNING Failed to wrap stderr. Console logs will not be captured.
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


✅ WandB run initialized: v5.4_strategy_a_targeted


## 5. データセットのアップロード

ローカルの `inputs/sft_processed/v5.4_targeted/train.json` をアップロードしてください。

In [6]:
import json
import shutil
from google.colab import files

print("v5.4_targeted train.json をアップロードしてください...")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.json'):
        target_path = '/content/v5.4_train.json'
        if filename != 'v5.4_train.json':
            shutil.move(filename, target_path)
        else:
            shutil.copy(filename, target_path)
        print(f"✅ アップロード完了: {target_path}")
        with open(target_path, 'r') as f:
            data = json.load(f)
        print(f"   データ件数: {len(data)} 件")
        break
else:
    print("⚠️ JSONファイルがアップロードされませんでした")

v5.4_targeted train.json をアップロードしてください...


Saving train.json to train.json
✅ アップロード完了: /content/v5.4_train.json
   データ件数: 17 件


## 6. 学習コード（v5.2ベースの追加SFT）

v5.2のLoRAアダプタを適用した状態から、さらにターゲットサンプルで追加学習します。

In [7]:
import random
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, Trainer, TrainerCallback
from peft import PeftModel, get_peft_model, LoraConfig

random.seed(SEED)

# ============================================================
# データセット読み込み
# ============================================================
def load_local_dataset(path: str) -> Dataset:
    """ローカルJSONファイルからデータセットを読み込む"""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_list(data)

# データセット読み込み
dataset_path = DATASET_ID.replace("LOCAL:", "")
ds = load_local_dataset(dataset_path)
print(f"✅ データセット読み込み完了: {len(ds)} 件")

✅ データセット読み込み完了: 17 件


In [8]:
# ============================================================
# モデル読み込み（v5.2アダプタをマージ）
# ============================================================
print(f"🔄 ベースモデル読み込み中: {BASE_MODEL_ID}")

# まずベースモデルを4bitで読み込み
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print("✅ ベースモデル読み込み完了")

# v5.2のLoRAアダプタを適用
print(f"🔄 v5.2アダプタ読み込み中: {V52_ADAPTER_ID}")
model = PeftModel.from_pretrained(model, V52_ADAPTER_ID)
print("✅ v5.2アダプタ適用完了")

# アダプタをベースモデルにマージ
print("🔄 アダプタをマージ中...")
model = model.merge_and_unload()
print("✅ マージ完了")

🔄 ベースモデル読み込み中: Qwen/Qwen3-4B-Instruct-2507
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/529M [00:00<?, ?B/s]

✅ v5.2アダプタ適用完了
🔄 アダプタをマージ中...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


✅ マージ完了


In [9]:
# ============================================================
# 新しいLoRAアダプタを追加（追加学習用）
# ============================================================
print("🔄 新しいLoRAアダプタを追加中...")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("✅ 新しいLoRAアダプタ追加完了")
model.print_trainable_parameters()

🔄 新しいLoRAアダプタを追加中...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2025.12.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ 新しいLoRAアダプタ追加完了
trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


In [10]:
# ============================================================
# データ前処理
# ============================================================
def format_messages(example):
    """messagesをテキストに変換"""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# messagesをテキストに変換
ds = ds.map(format_messages)

# トークナイズ
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

ds = ds.map(tokenize, remove_columns=["messages", "text", "id", "metadata"])
print(f"✅ データ前処理完了: {len(ds)} 件")

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

✅ データ前処理完了: 17 件


In [11]:
# ============================================================
# SFTトレーナー設定
# ============================================================
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUT_LORA_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    max_steps=MAX_STEPS,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="wandb",
)

print(f"✅ TrainingArguments設定完了")
print(f"   Learning Rate: {LR}")
print(f"   Epochs: {NUM_TRAIN_EPOCHS}")
print(f"   Batch Size: {PER_DEVICE_TRAIN_BATCH_SIZE}")

✅ TrainingArguments設定完了
   Learning Rate: 1e-06
   Epochs: 1
   Batch Size: 2


In [19]:
# ============================================================
# モデル再読み込みと学習実行
# ============================================================
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer
from peft import PeftModel
from unsloth import FastLanguageModel


# 1. ベースモデルをUnslothで読み込み（4bit）
print(f"🔄 ベースモデル読み込み中: {BASE_MODEL_ID}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

# 2. v5.2アダプタを読み込み（Trainableとして設定）
# マージせずにアダプタをそのまま学習対象にします
print(f"🔄 v5.2アダプタ読み込み中: {V52_ADAPTER_ID}")
model = PeftModel.from_pretrained(
    model,
    V52_ADAPTER_ID,
    is_trainable=True
)
print("✅ モデル準備完了 (Mergeなし, Adapter Fine-tuning mode)")

# ★ 追加修正: 入力の勾配計算を有効化（QLoRAでのエラー回避に必須）
model.enable_input_require_grads()
print("ℹ️ Input grads enabled.")

# ★ 追加修正: モデル側で明示的に Gradient Checkpointing を無効化
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
    print("ℹ️ Gradient Checkpointing disabled (model method).")
elif hasattr(model, "base_model") and hasattr(model.base_model, "gradient_checkpointing_disable"):
    model.base_model.gradient_checkpointing_disable()
    print("ℹ️ Gradient Checkpointing disabled (base_model method).")

# 設定ファイルも念のため更新
model.config.use_cache = True
model.config.gradient_checkpointing = False

# 学習対象パラメータの確認
model.print_trainable_parameters()
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Gradient Checkpointingを引数レベルでも無効化
if hasattr(training_args, "gradient_checkpointing"):
    training_args.gradient_checkpointing = False

# トレーナー設定（tokenizerを明示）
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
    max_seq_length=MAX_SEQ_LEN,
)

print("🚀 学習開始...")
print(f"   データ件数: {len(ds)} 件")
# 予想ステップ数
print(f"   予想ステップ数: {len(ds) // PER_DEVICE_TRAIN_BATCH_SIZE // GRAD_ACCUM * NUM_TRAIN_EPOCHS}")
trainer.train()
print("✅ 学習完了!")

🔄 ベースモデル読み込み中: Qwen/Qwen3-4B-Instruct-2507
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 v5.2アダプタ読み込み中: kmd2525/qwen3-4b-structured-output-lora-v5.2
✅ モデル準備完了 (Mergeなし, Adapter Fine-tuning mode)
ℹ️ Input grads enabled.
ℹ️ Gradient Checkpointing disabled (model method).
trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 学習開始...
   データ件数: 17 件
   予想ステップ数: 8


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 17 | Num Epochs = 1 | Total steps = 9
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Step,Training Loss
1,1.374600
2,1.189100
3,1.468300
4,1.135500
5,1.361100
6,1.321500
7,1.154200
8,1.146100
9,1.232400


train/epoch,▁▂▃▄▅▅▆▇██
train/global_step,▁▂▃▄▅▅▆▇██
train/grad_norm,▆██▄▇▅▆▁▄
train/learning_rate,▁██▇▆▅▃▂▁
train/loss,▆▂█▁▆▅▁▁▃
total_flos,129756165098496.0
train/epoch,1
train/global_step,9
train/grad_norm,2.86989
train/learning_rate,0.0
train/loss,1.2324


✅ 学習完了!


In [20]:
# ============================================================
# LoRAアダプタの保存
# ============================================================
print(f"💾 アダプタを保存中: {OUT_LORA_DIR}")

model.save_pretrained(OUT_LORA_DIR)
tokenizer.save_pretrained(OUT_LORA_DIR)

print("✅ 保存完了")
print(f"📁 保存先: {OUT_LORA_DIR}")

# 保存されたファイルを確認
import os
print("\n📄 保存されたファイル:")
for f in os.listdir(OUT_LORA_DIR):
    print(f"   - {f}")

💾 アダプタを保存中: /content/lora_structeval_t_qwen3_4b_v5.4
✅ 保存完了
📁 保存先: /content/lora_structeval_t_qwen3_4b_v5.4

📄 保存されたファイル:
   - adapter_config.json
   - checkpoint-9
   - merges.txt
   - special_tokens_map.json
   - adapter_model.safetensors
   - vocab.json
   - chat_template.jinja
   - tokenizer_config.json
   - tokenizer.json
   - README.md
   - added_tokens.json


## 7. README.md 作成

In [21]:
readme_md = f"""---
license: apache-2.0
base_model: {BASE_MODEL_ID}
library_name: peft
tags:
  - lora
  - structured-output
  - sft
  - v5.4
  - strategy-a
  - targeted-learning
---

# {TITLE_LINE}

LoRA adapter for structured output generation (v5.4 Strategy A).

## Strategy A: v5.2 Base + Low LR Targeted Learning

This adapter was created by:
1. Loading v5.2 trained adapter (`{V52_ADAPTER_ID}`)
2. Merging into base model
3. Additional SFT with 17 targeted samples (TOML 14, XML 3)
4. Very low learning rate (1e-6) to preserve v5.2 learning

## Training Details

- Base: v5.2 merged model (94.7% accuracy)
- Additional data: 17 targeted samples
- Learning rate: 1e-6 (very low)
- Epochs: 1
- LoRA r: {LORA_R}
- LoRA alpha: {LORA_ALPHA}

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = "{BASE_MODEL_ID}"
adapter = "your_id/{TITLE_LINE}"

tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(
    base,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter)
```

## Note

This model builds upon v5.2 with targeted fixes for:
- TOML inline table syntax
- TOML array table syntax
- XML special character escaping
"""

readme_path = os.path.join(OUT_LORA_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_md)

print(f"✅ README.md written to: {readme_path}")

✅ README.md written to: /content/lora_structeval_t_qwen3_4b_v5.4/README.md


## 8. HuggingFace へアップロード

In [22]:
import fnmatch
import shutil
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi()
LORA_SAVE_DIR = Path(OUT_LORA_DIR)
HF_REPO_ID = _getenv("HF_REPO_ID", f"kmd2525/{TITLE_LINE}")
IS_PRIVATE = _getenv("HF_PRIVATE", "1") in ("1", "true", "True")

# 必須ファイルの存在確認
required_files = {
    "adapter_config.json",
    "README.md",
}

present = {p.name for p in LORA_SAVE_DIR.iterdir() if p.is_file()}
missing = [f for f in required_files if f not in present]

if not any(f.startswith("adapter_model.") for f in present):
    missing.append("adapter_model.(safetensors|bin)")

if missing:
    raise RuntimeError(
        "アップロードを中止しました。\n"
        "以下の必須ファイルが見つかりません:\n"
        + "\n".join(f"- {m}" for m in missing)
    )

print("✅ 必須ファイルの確認が完了しました。")

# アップロード対象の選別
ALLOW_PATTERNS = [
    "README.md",
    "adapter_config.json",
    "adapter_model.*",
    "tokenizer.*",
    "special_tokens_map.json",
    "*.json",
]

def is_allowed(name: str) -> bool:
    return any(fnmatch.fnmatch(name, pat) for pat in ALLOW_PATTERNS)

STAGE_DIR = Path("/content/hf_upload_stage")

if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True)

for p in LORA_SAVE_DIR.iterdir():
    if p.is_file() and is_allowed(p.name):
        (STAGE_DIR / p.name).write_bytes(p.read_bytes())

print("📦 アップロード対象ファイル:", [p.name for p in STAGE_DIR.iterdir()])

# リポジトリ作成とアップロード
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=IS_PRIVATE,
    token=MY_TOKEN,
)

api.upload_folder(
    folder_path=str(STAGE_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload v5.4 Strategy A LoRA adapter",
    token=MY_TOKEN,
)

print("✅ アップロードが正常に完了しました。")
print(f"URL: https://huggingface.co/{HF_REPO_ID}")

✅ 必須ファイルの確認が完了しました。
📦 アップロード対象ファイル: ['adapter_config.json', 'special_tokens_map.json', 'adapter_model.safetensors', 'vocab.json', 'tokenizer_config.json', 'tokenizer.json', 'README.md', 'added_tokens.json']


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...load_stage/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

  ...adapter_model.safetensors:   0%|          | 61.5kB /  529MB            

✅ アップロードが正常に完了しました。
URL: https://huggingface.co/kmd2525/qwen3-4b-structured-output-lora-v5.4


In [23]:
# WandB終了
wandb.finish()

# 完了通知音
from google.colab import output

js_code = """
(function() {
  try {
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var osc = ctx.createOscillator();
    var gain = ctx.createGain();
    osc.connect(gain);
    gain.connect(ctx.destination);
    osc.type = 'sine';
    osc.frequency.value = 523.25;
    gain.gain.setValueAtTime(0.5, ctx.currentTime);
    gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + 1.5);
    osc.start();
    osc.stop(ctx.currentTime + 1.5);
  } catch (e) { console.log("Audio play failed", e); }
})();
"""

output.eval_js(js_code)

print("🎉 v5.4 Strategy A 学習・アップロード完了!")

🎉 v5.4 Strategy A 学習・アップロード完了!


## 9. Colabインスタンスの削除（課金停止）

学習が完了したら、以下のコードを実行してColabインスタンスを削除し、課金を停止できます。

In [25]:
from google.colab import runtime
runtime.unassign()